In [68]:
import pandas as pd
import re
import nltk
import os
from pathlib import Path
import os



unc_remove_re = re.compile(r'\W+')
# Old: corp_re = re.compile('( (group|holding(s)?( co)?|inc(orporated)?|ltd|l ?l? ?[cp]|co(rp(oration)?|mpany)?|s[ae]|plc))+$')
# added flags=re.IGNORECASE to make corp_re case insensitive
corp_re = re.compile(r'( (group|holding(s)?( co)?|inc(orporated)?|ltd|l ?l? ?[cp]|co(rp(oration)?|mpany)?|s[ae]|plc|fund|trust|spv))+$', flags=re.IGNORECASE)
and_re = re.compile(' & ')
punc1_re = re.compile(r'(?<=\S)[\'’´\.](?=\S)')
# added number removal to punc2_re
punc2_re = re.compile(r'[\s\.,:;/\'"`´‘’“”\(\)\[\]\{\}_—\-?$=!#0-9]+') # Used in basicHash for number removal

STOPWORDS = nltk.corpus.stopwords.words('english')
STOPWORDS.remove("am")
STOPWORDS.remove("up")
STOPWORDS.remove("in")
STOPWORDS.remove("on")
STOPWORDS.remove("all")
STOPWORDS.remove("any")
STOPWORDS.remove("most")
STOPWORDS.remove("no")
STOPWORDS.remove("nor")
STOPWORDS.remove("own")
STOPWORDS.remove("same")
STOPWORDS.remove("so")
STOPWORDS.remove("very")
STOPWORDS.remove("s")
STOPWORDS.remove("t")
STOPWORDS.remove("d")
STOPWORDS.remove("ll")
STOPWORDS.remove("m")
STOPWORDS.remove("o")
STOPWORDS.remove("re")
STOPWORDS.remove("ve")
STOPWORDS.remove("y")

stopword_re_str = r""
for word in STOPWORDS:
	stopword_re_str += r'\b' + word + r'\b|'
stopword_re = re.compile(stopword_re_str[:-1]) # The negative 1 is for the fencepost |


# Function to calculate longest common substring, from https://www.geeksforgeeks.org/print-longest-common-substring/
# function to find and print 
# the longest common substring of
# X[0..m-1] and Y[0..n-1]
def get_longest_common_substring(X, Y, m, n):
 
    # Create a table to store lengths of
    # longest common suffixes of substrings.
    # Note that LCSuff[i][j] contains length
    # of longest common suffix of X[0..i-1] and
    # Y[0..j-1]. The first row and first
    # column entries have no logical meaning,
    # they are used only for simplicity of program
    LCSuff = [[0 for i in range(n + 1)]
                 for j in range(m + 1)]
 
    # To store length of the
    # longest common substring
    length = 0
 
    # To store the index of the cell
    # which contains the maximum value.
    # This cell's index helps in building
    # up the longest common substring
    # from right to left.
    row, col = 0, 0
 
    # Following steps build LCSuff[m+1][n+1]
    # in bottom up fashion.
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0:
                LCSuff[i][j] = 0
            elif X[i - 1] == Y[j - 1]:
                LCSuff[i][j] = LCSuff[i - 1][j - 1] + 1
                if length < LCSuff[i][j]:
                    length = LCSuff[i][j]
                    row = i
                    col = j
            else:
                LCSuff[i][j] = 0
 
    # if true, then no common substring exists
    if length == 0:
        return ""
 
    # allocate space for the longest
    # common substring
    resultStr = ['0'] * length
 
    # traverse up diagonally form the
    # (row, col) cell until LCSuff[row][col] != 0
    while LCSuff[row][col] != 0:
        length -= 1
        resultStr[length] = X[row - 1] # or Y[col-1]
 
        # move diagonally up to previous cell
        row -= 1
        col -= 1
 
    # required longest common substring
    longest_common_substring = ''.join(resultStr)

    return longest_common_substring


# Function from Brad Hackinen's NAMA
def basicHash(s):
    '''
    A simple case and puctuation-insensitive hash
    '''
    s = s.lower()
    s = re.sub(and_re,' and ',s)
    s = re.sub(punc1_re,'',s)
    
    # Remove numbers too to avoid issues with things like "3M"
    # Modified from Brad Hackinen's version
    punc2_re = re.compile(r'[\s\.,:;/\'"`´‘’“”\(\)\[\]\{\}_—\-?$=!#0-9]+')
    s = re.sub(punc2_re,' ',s)
    s = s.strip()

    return s

# Function from Brad Hackinen's NAMA
def corpHash(s):
    '''
    A hash function for corporate subsidiaries
    Insensitive to
        -case & punctation
        -'the' prefix
        -common corporation suffixes, including 'holding co'
    '''
    s = basicHash(s)
    if s.startswith('the '):
        s = s[4:]

    s = re.sub(corp_re,'',s,count=1)
    
    # Remove stopwords here
    # Prevents accidental removal of words in corporate suffixes or prefixes
    s = re.sub(stopword_re, ' ', s)

    # Clean up excess spaces left by stopword removal
    s = re.sub(r'\s+', ' ', s)
    
    return s.strip()

    #return s

# function to clean org names
def clean_fin_org_names1(name):
    if name is None or not isinstance(name, str) or name == "NA":
        return ""
    else:
        # James strip metadata from name
        name = name.split(',')[0]
        name = re.sub(" [0-9]* [k|m]b pdf","",name)

        # name = name.translate(corp_simplify_utils.STR_TABLE)
        # comment out this line for now since corp_simplify_utils is not imported
        # TO DO: import corp_simplify_utils if needed, and ask for access if it's private
        name = re.sub(stopword_re, '', name.lower())
        
        return corpHash(name)
    
# UPDATED VERSION: function to clean org names
def clean_fin_org_names(name):
    '''
    Cleans an individual organization name string.
    '''
    if name is None or not isinstance(name, str) or name == "NA":
        return ""
    
    name = name.lower()
    # The split(',') has been removed to prevent prematurely cutting off names like "Fund I, LP" 
    # Was having issues with funds being cut off incorrectly
    # also removed name = re.sub(stopword_re, '', name.lower()) because stopwords are now removed in corpHash
    name = re.sub(r"\s+[0-9]*\s+[k|m]b\s+pdf\s*$", "", name)


    return corpHash(name)

In [69]:
# Locate data directory and read in data files

current_dir = Path(os.getcwd()).parent
#print(current_dir)
data_dir = current_dir / 'data'
data_dir = data_dir.resolve()
#print(data_dir)

try:
    compustat_df = pd.read_csv(data_dir / 'CompustatNames.csv')
    cik_df = pd.read_csv(data_dir / 'CIK.csv')
    fdic_df = pd.read_csv(data_dir / 'FDIC_clean.csv') # Using your 'FDIC_clean.csv'
    sec_df = pd.read_csv(data_dir / 'SEC_Institutions.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()

In [70]:
# cleaning and standardizing organization names
compustat_df['std_name'] = compustat_df['conm'].apply(clean_fin_org_names)
fdic_df['std_name'] = fdic_df['NAME'].apply(clean_fin_org_names)
sec_df['std_name'] = sec_df['Name'].apply(clean_fin_org_names)
cik_df['std_name'] = cik_df['company_name'].apply(clean_fin_org_names)

print(cik_df[['company_name', 'std_name']].head(10))
print(sec_df[['Name', 'std_name']].head(10))

                         company_name                     std_name
0                              !J INC                            j
1       #1 A LIFESAFER HOLDINGS, INC.                    lifesafer
2  #1 ARIZONA DISCOUNT PROPERTIES LLC  arizona discount properties
3                   #1 PAINTBALL CORP                    paintball
4                               $ LLC                          llc
5                          $AVY, INC.                          avy
6                 & S MEDIA GROUP LLC                    & s media
7             &TV COMMUNICATIONS INC.           &tv communications
8   &VEST DOMESTIC FUND II KPIV, L.P.  &vest domestic fund ii kpiv
9           &VEST DOMESTIC FUND II LP       &vest domestic fund ii
                                             Name  \
0                        Agilent Technologies Inc   
1                                       Alcoa Inc   
2                Asia Automotive Acquisition Corp   
3                              Asia Broadband Inc  

In [71]:
print("\n--- SEC DF Output ---\n") # adding a visible break to look at dfs
print(sec_df.head(10))
print("\n--- CIK DF Output ---\n") 
print(cik_df.head(10))
print("\n--- Compustat DF Output ---\n")
print(compustat_df.head(10))
print("\n--- FDIC DF Output ---\n") 
print(fdic_df.head(10))


--- SEC DF Output ---

   index      CIK Ticker                                            Name  \
0      0  1090872      A                        Agilent Technologies Inc   
1      1     4281     AA                                       Alcoa Inc   
2      2  1332552  AAACU                Asia Automotive Acquisition Corp   
3      3  1287145   AABB                              Asia Broadband Inc   
4      4  1024015   AABC                      Access Anytime Bancorp Inc   
5      5  1099290    AAC  Sinocoking Coal & Coke Chemical Industries Inc   
6      6  1264707   AACC                   Asset Acceptance Capital Corp   
7      7   849116   AACE                            Ace Cash Express Inc   
8      8  1409430   AAGC                          All American Gold Corp   
9      9   948846    AAI                            Airtran Holdings Inc   

  Exchange     SIC Business Incorporated          IRS  \
0     NYSE  3825.0       CA           DE  770518772.0   
1     NYSE  3350.0       

In [72]:
# Test if CIK is already in SEC
sec_ciks = set(sec_df['CIK'].dropna())
cik_ciks = set(cik_df['cik'].dropna())

print(f"CIKs in sec_df: {len(sec_ciks)}")
print(f"CIKs in cik_df: {len(cik_ciks)}")
print(f"Is SEC_Institutions.csv a subset of CIK.csv? {sec_ciks.issubset(cik_ciks)}")

compustat_ciks = set(compustat_df['cik'].dropna())
print(f"CIKs in compustat_df: {len(compustat_ciks)}")
print(f"Is CompustatNames.csv a subset of CIK.csv? {compustat_ciks.issubset(cik_ciks)}")


CIKs in sec_df: 13737
CIKs in cik_df: 806225
Is SEC_Institutions.csv a subset of CIK.csv? True
CIKs in compustat_df: 12835
Is CompustatNames.csv a subset of CIK.csv? False


SEC_Institutions.csv is a subset of CIK.csv --- > Don't need to merge SEC into the crosswalk. 

In [73]:
# renaming columns for consistency
compustat_temp = compustat_df[['std_name', 'conm', 'tic', 'cusip', 'cik']].rename(columns={'conm': 'raw_name'})
compustat_temp['source'] = 'compustat'

fdic_temp = fdic_df[['std_name', 'NAME']].rename(columns={'NAME': 'raw_name'})
fdic_temp['source'] = 'fdic'

cik_temp = cik_df[['std_name', 'company_name', 'cik']].rename(columns={'company_name': 'raw_name'})
cik_temp['source'] = 'cik'

# Combine all into one long dataframe
all_names_df = pd.concat([compustat_temp, fdic_temp, cik_temp], ignore_index=True)

# Drop any rows where cleaning failed (no std_name)
all_names_df = all_names_df.dropna(subset=['std_name'])
# Remove any empty std_name entries
all_names_df = all_names_df[all_names_df['std_name'] != ""]

# Cleaning up CIKs and FED_RSSD to be strings without decimal points
for col in ['CIK', 'FED_RSSD']:
    if col in all_names_df.columns:
        # Convert to string after converting to int to remove any decimal points
        all_names_df[col] = all_names_df[col].dropna().astype(float).astype(int).astype(str)

print(f"Total entries to match: {len(all_names_df)}")
print(all_names_df.head(10))
print(all_names_df.shape)

Total entries to match: 915183
                   std_name                      raw_name     tic      cusip  \
0                       aar                      AAR CORP     AIR  000361105   
1    adc telecommunications    ADC TELECOMMUNICATIONS INC  ADCT.1  000886309   
2         american airlines   AMERICAN AIRLINES GROUP INC     AAL  02376R102   
3        ceco environmental       CECO ENVIRONMENTAL CORP    CECE  125141101   
4  asa gold precious metals  ASA GOLD AND PRECIOUS METALS     ASA  G3156P103   
5                       avx                      AVX CORP     AVX  002444107   
6     pinnacle west capital    PINNACLE WEST CAPITAL CORP     PNW  723484101   
7                      prog             PROG HOLDINGS INC     PRG  74319R101   
8       abbott laboratories           ABBOTT LABORATORIES     ABT  002824100   
9                 servidyne                 SERVIDYNE INC  SERV.1  81765M106   

         cik     source  
0     1750.0  compustat  
1    61478.0  compustat  
2     6201

In [74]:
grouped_by_cik_id = all_names_df.groupby('cik')
confident_matches = []

for cik_value, group in grouped_by_cik_id:
    if len(group) > 1:
        # Aggregate the data based on cik
        keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            # Now aggregate the std_name to see all variations found for cik
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['raw_name'].dropna().unique()),
            'sources': ','.join(group['source'].unique())
        }
        confident_matches.append(keys)
        
pd.set_option('display.max_colwidth', None)
confident_crosswalk_id_based = pd.DataFrame(confident_matches)
print(f"Found {len(confident_crosswalk_id_based)} confident entity clusters.")
print(confident_crosswalk_id_based.head(100))


Found 56750 confident entity clusters.
         cik  \
0   [1750.0]   
1   [1800.0]   
2   [1841.0]   
3   [1853.0]   
4   [1860.0]   
5   [1904.0]   
6   [1918.0]   
7   [1923.0]   
8   [1961.0]   
9   [2034.0]   
10  [2062.0]   
11  [2093.0]   
12  [2098.0]   
13  [2110.0]   
14  [2178.0]   
15  [2186.0]   
16  [2230.0]   
17  [2310.0]   
18  [2380.0]   
19  [2488.0]   
20  [2491.0]   
21  [2554.0]   
22  [2601.0]   
23  [2646.0]   
24  [2648.0]   
25  [2663.0]   
26  [2664.0]   
27  [2691.0]   
28  [2768.0]   
29  [2781.0]   
30  [2784.0]   
31  [2809.0]   
32  [2969.0]   
33  [3000.0]   
34  [3116.0]   
35  [3124.0]   
36  [3146.0]   
37  [3153.0]   
38  [3197.0]   
39  [3270.0]   
40  [3327.0]   
41  [3370.0]   
42  [3398.0]   
43  [3453.0]   
44  [3499.0]   
45  [3520.0]   
46  [3521.0]   
47  [3530.0]   
48  [3545.0]   
49  [3570.0]   
50  [3588.0]   
51  [3673.0]   
52  [3683.0]   
53  [3721.0]   
54  [3794.0]   
55  [3798.0]   
56  [3845.0]   
57  [3906.0]   
58  [3952.0]   
5

Able to match 56752 companies based on CIK id.

In [75]:
# Find all the rows where there is duplicate CIK in one of the dataframes
# In other words, there are multiple aliases, but coming from the same source

confident_crosswalk_id_based[confident_crosswalk_id_based['aliases'].str.contains('\|') & ~confident_crosswalk_id_based['sources'].str.contains(r',')].head(20)

,cik,standardized_names,aliases,sources
2,[1841.0],abel noser corp bd|abel noser,ABEL NOSER CORP /BD|ABEL/NOSER CORP.,cik
3,[1853.0],aberdeen idaho mining|motivnation,"ABERDEEN IDAHO MINING CO|MOTIVNATION, INC.",cik
4,[1860.0],thomson richard william bd|thomson richard william,"THOMSON RICHARD WILLIAM /BD|THOMSON, RICHARD WILLIAM",cik
5,[1904.0],abraham co inc bd|abraham|abraham securities,"ABRAHAM & CO INC /BD|ABRAHAM & CO., INC.|ABRAHAM SECURITIES CORPORATION",cik
6,[1918.0],abrams allan edward|homeland securities financial services|merchanthouse securities|wizer financial,"ABRAMS, ALLAN EDWARD|HOMELAND SECURITIES FINANCIAL SERVICES GROUP, INC.|THE MERCHANTHOUSE SECURITIES, INC.|WIZER FINANCIAL. INC.",cik
11,[2093.0],acme metals inc de|acme metals,ACME METALS INC /DE/|ACME METALS INC/,cik
13,[2110.0],acorn investment|columbia acorn|liberty acorn,ACORN INVESTMENT TRUST|COLUMBIA ACORN TRUST|LIBERTY ACORN TRUST,cik
17,[2310.0],am international|multigraphics,AM INTERNATIONAL INC|MULTIGRAPHICS INC,cik
18,[2380.0],administrative data management corp ta|foresters investor services inc ta,ADMINISTRATIVE DATA MANAGEMENT CORP /TA|ADMINISTRATIVE DATA MANAGEMENT CORP /TA|FORESTERS INVESTOR SERVICES INC/TA,cik
21,[2554.0],aei securities inc bd|aei securities,"AEI SECURITIES INC /BD|AEI SECURITIES, INC.",cik


There appears to be many aliases for the same CIK ID in the cik.csv file

In [76]:
# need to isolate remaining data that couldn't be matched by CIK
# checks the size of each group by CIK
cik_group_sizes = all_names_df.groupby('cik')['cik'].transform('size')
processed_rows_mask = (cik_group_sizes > 1)
remaining_df = all_names_df[~processed_rows_mask].copy()

total_rows = len(all_names_df)
processed_rows_count = processed_rows_mask.sum()
remaining_rows_count = len(remaining_df)
print(f"Total rows: {total_rows}")
print(f"Processed rows (matched by CIK): {processed_rows_count}")
print(f"Remaining rows to process: {remaining_rows_count}")

Total rows: 915183
Processed rows (matched by CIK): 133362
Remaining rows to process: 781821


In [77]:
# Remaining rows to process will go through other matching methods including fuzzy matching and regex-based matching

grouped_by_std_name = remaining_df.groupby('std_name')
confident_matches_std_name = []

for name, group in grouped_by_std_name:
    # A "match" means this std_name appeared in more than one row
    if len(group) > 1:
        # Aggregate all unique keys and aliases
        keys = {
            'std_name': name,
            'aliases': '|'.join(group['raw_name'].dropna().unique()),
            'sources': ','.join(group['source'].unique()),
            'CIK': group['cik'].dropna().unique().tolist(),
        }
        confident_matches_std_name.append(keys)
pd.set_option('display.max_rows', None)
confident_crosswalk_std_name = pd.DataFrame(confident_matches_std_name)
print(f"  Found {len(confident_crosswalk_std_name)} additional clusters based on std_name.")
print(confident_crosswalk_std_name.head(100))

  Found 27845 additional clusters based on std_name.
                                                         std_name  \
0                                                   &p investment   
1                                            &q alternative yield   
2                                    aacp china venture investors   
3                                                            aaeb   
4                                                             aai   
5                                                  aames mortgage   
6                                       aames mortgage investment   
7                                                          aarifs   
8                                              aasmundstad eric k   
9                                                             aat   
10                                                       aavcf pf   
11                    ab abbott private equity investors delaware   
12                    ab abbott private equity sol